# Part 3: Building a dataset

This notebook generates a random neural network, synthesizes it with hls4ml, and reads the synthesis record back with `rule4ml`. Many such records make a training dataset, like [`wa-hls4ml`](https://huggingface.co/datasets/fastmachinelearning/wa-hls4ml).

It runs from the `notebooks` folder of a clone of the repository, with the data generation and notebook dependencies installed from the repository root:

```bash
uv sync --group datagen --group notebooks           # with uv
pip install -e . --group datagen --group notebooks  # with pip
```

Synthesis also needs the AMD tools. This notebook assumes the Vitis backend 2024.2 is installed.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))  # data_gen lives at the repository root

## Generating a network

`GeneratorSettings` holds the ranges a random network is drawn from: input size, number of layers, neurons per layer, output size, activations, and the probabilities of bias, batch normalization, dropout and skip connections.

Calling `generate_fc_network` draws a fully connected Keras model from them.

In [ ]:
import numpy as np

from data_gen.nn_gen import (
    GeneratorSettings,
    IntRange,
    Power2Range,
    generate_fc_network,
)

settings = GeneratorSettings(
    input_range=Power2Range(16, 64),
    layer_range=IntRange(2, 4),
    neuron_range=Power2Range(8, 64),
    output_range=IntRange(1, 10),
)
model = generate_fc_network(settings, rng=np.random.default_rng(0))
model.summary()

## Converting with hls4ml

The HLS config sets the precision, reuse factor and strategy, here for the whole model. hls4ml converts the model to an HLS project and compiles its C++ emulation. This step needs no AMD tools, so it checks the conversion before a long synthesis.

In [ ]:
import hls4ml

from data_gen.nn_synth import keras_to_hls4ml
from rule4ml.parsers.utils import get_part_from_board

hls_config = hls4ml.utils.config_from_keras_model(model, granularity="name")
hls_config["Model"].update(
    {"Precision": "ap_fixed<8,3>", "ReuseFactor": 4, "Strategy": "Resource"}
)
part_number = get_part_from_board("zcu102")

hls_model = keras_to_hls4ml(
    model,
    "projects/conversion_check",
    hls_config,
    clock_period=5,
    part_number=part_number,
    io_type="io_parallel",
    backend="vitis",
)

## Synthesizing and reading the record

`synthesize_keras_model` converts the model again, runs HLS and logic synthesis, and returns a record: the model and HLS configs, the resource reports of HLS and of logic synthesis, the latency report, and the tool versions.

`dataset.prepare` turns saved records into the `.npz` format that training reads.

Set `XILINX_PATH` to the folder holding Vitis/Vivado. Synthesis can take minutes to hours depending on the model.

In [ ]:
import json

from data_gen.nn_synth import synthesize_keras_model
from rule4ml import dataset

XILINX_PATH = "/tools/Xilinx"
XILINX_BACKEND = "Vitis"
XILINX_VERSION = "2024.2"
BACKEND_PATH = os.path.join(XILINX_PATH, XILINX_BACKEND, XILINX_VERSION)

if os.path.isdir(BACKEND_PATH):
    record = synthesize_keras_model(
        model,
        "projects",
        hls_config,
        clock_period=5,
        part_number=part_number,
        io_type="io_parallel",
        backend=XILINX_BACKEND,
        backend_version=XILINX_VERSION,
        xilinx_path=XILINX_PATH,
        csim=False,
        synth=True,
        vsynth=True,
    )

    os.makedirs("records", exist_ok=True)
    with open("records/record.json", "w") as json_file:
        json.dump([record], json_file)

    dataset.prepare(["records/record.json"], "records/records.npz")
    records = dataset.load("records/records.npz")

    print("LUT:", records.targets(["lut"], "resource_report")[0])
    print("Cycles:", records.targets(["cycles_min", "cycles_max"])[0])
    print(
        "Global features:",
        dict(zip(records.global_names[:6], records.global_values[0][:6].tolist())),
    )
else:
    print(
        f"{XILINX_BACKEND} {XILINX_VERSION} not found in {XILINX_PATH}, skipping synthesis"
    )

## Larger datasets

`data_gen.nn_synth.parallel_rnd_synthesis` draws and synthesizes many networks per job, with the HLS settings drawn from `SynthSettings`. You can run several jobs in parallel, for example with `multiprocessing.Pool`, then pass all the saved JSON files to `dataset.prepare`.